In [1]:
import os

from glob import glob

import numpy as np
import dask.array as da
import zarr
from stack_to_multiscale_ngff.h5_nested_store3 import H5_Nested_Store
from numcodecs import Blosc
from scipy.ndimage import gaussian_filter1d
from scipy.io import loadmat
import scipy.io as sio

In [2]:
def read_omehans(path_to_omehans, scale=None):
    location = os.path.join(path_to_omehans, f'scale{scale}' if scale else "")
    store = H5_Nested_Store(location)
    zarray = zarr.open(store)
    dask_zarray = da.array(zarray)
    return dask_zarray

In [18]:
def _unwrap_obj(x):
    while isinstance(x, np.ndarray) and x.dtype == object and x.size == 1:
        x = x.reshape(-1)[0]
    return x

def parse_A2D_list(transforms_mat) -> list:
    """
    Input: dict returned by scipy.io.loadmat(..., struct_as_record=False, squeeze_me=True)
    Output: list of 3x3 float32 affines [A2D_ch0, A2D_ch1, ..., A2D_chN]
    Works whether 'Transforms' is:
      - an object array/list of mat_structs each with .A2D
      - a single mat_struct whose .A2D is an array/cell of matrices
    """
    T = transforms_mat['Transforms']
    T = np.squeeze(T)

    mats = []

    # Case A: array/list of channel structs
    if isinstance(T, (list, tuple)) or (isinstance(T, np.ndarray) and T.dtype == object and T.ndim == 1):
        for node in (T if isinstance(T, (list, tuple)) else list(T)):
            node = _unwrap_obj(node)
            A2D = getattr(node, 'A2D', None)
            if A2D is None:
                A2D = node['A2D']  # rare, but some mats keep dict-style
            A2D = _unwrap_obj(A2D)
            A2D = np.asarray(A2D, dtype=np.float32)
            if A2D.shape == (2,3):
                A2D = np.vstack([A2D, [0,0,1]]).astype(np.float32)
            if A2D.shape != (3,3):
                raise ValueError(f"Unexpected A2D shape {A2D.shape}")
            mats.append(A2D)

    else:
        # Case B: single struct with a field A2D that is a collection of per-channel matrices
        node = _unwrap_obj(T)
        A_all = getattr(node, 'A2D', None)
        if A_all is None:
            A_all = node['A2D']
        A_all = np.squeeze(np.asarray(A_all, dtype=object))
        # normalize to 1D iterable of objects
        if A_all.ndim == 0:
            A_iter = [A_all.item()]
        elif A_all.ndim == 1:
            A_iter = list(A_all)
        elif A_all.shape[0] == 1:
            A_iter = list(A_all[0])
        elif A_all.shape[1] == 1:
            A_iter = list(A_all[:,0])
        else:
            A_iter = list(A_all)

        for a in A_iter:
            a = _unwrap_obj(a)
            M = np.asarray(a, dtype=np.float32)
            if M.shape == (2,3):
                M = np.vstack([M, [0,0,1]]).astype(np.float32)
            if M.shape != (3,3):
                raise ValueError(f"Unexpected A2D shape {M.shape}")
            mats.append(M)

    if not mats:
        raise ValueError("No A2D matrices found in 'Transforms'")

    return mats


In [6]:
# Transforms and other .mat info path
import scipy.io as sio
transforms_matlab_path = "/bil/proj/rf1hillman/HOLiS_NPBB328_Cortex/Matlab_info/NPBB328_colorMerge_transforms.mat"   
matrices_matlab_path = "/bil/proj/rf1hillman/HOLiS_NPBB328_Cortex/Matlab_info/NPBB328_SimulationMatrices_equalPower.mat"
transforms = sio.loadmat(transforms_matlab_path)
matrices = sio.loadmat(matrices_matlab_path)

In [20]:
FOVcrop_raw = transforms["FOVcrop"]

In [12]:
laser_power_at_sample = transforms['FOVcrop']

In [26]:
print(laser_power_at_sample)

[[(array([[65, 19]], dtype=uint8), array([[98, 60]], dtype=uint8))]]


In [3]:
#vol_unmixed=read_omehans('/bil/proj/rf1hillman/results/2025_04_18_NPBB328_Slab06_test/test_Sept2025/flis/out/NPBB328-Slab06-scanForPeterAndIana-run002-Exc-488nm-561nm-594nm-660nm_HiCAM_FLUO_1875-ST-088.fli_unmixed/omehans/')

In [4]:
#vol_unmixedArray = vol_unmixed.compute()

In [5]:
#print(vol_unmixed.shape)

In [3]:
slab = 6
z_slices = 13
sample_dir= '/bil/proj/rf1hillman/HOLiS_NPBB328_Cortex/Slab6/2025_08_22_HOLiS_NPBB328_Cortex_Slab06/'
corrections_dir = '/bil/proj/rf1hillman/HOLiS_NPBB328_Cortex/Slab6/2025_08_22_HOLiS_NPBB328_Cortex_Slab06_corrections'
output_dir = '/bil/proj/rf1hillman/results/NPBB328_Cortex/Slab6/correction_masks'
output_dir_omehans = '/bil/proj/rf1hillman/results/NPBB328_Cortex/Slab6/correction_omehans'

## Nuclei

### Background load

In [33]:
# Create oemhans
# file pattern: f'NPBB328-Cortex-Slab06-run*-z*-y000-darkFrames_HiCAM FLUO_1875-ST-272.fli.zst'
# !python queue_hicam_files.py

>>>>>>>>>>> Files: 18
Input file: /bil/proj/rf1hillman/HOLiS_NPBB328_Cortex/Slab6/2025_08_22_HOLiS_NPBB328_Cortex_Slab06/NPBB328-Cortex-Slab06-run388-z05-y000-darkFrames_HiCAM FLUO_1875-ST-272.fli.zst
job_script_path: /bil/proj/rf1hillman/results/NPBB328_Cortex/Slab6/correction_omehans/job_NPBB328-Cortex-Slab06-run388-z05-y000-darkFrames_HiCAM FLUO_1875-ST-272.sh
Submitted batch job 976075
Submitted job for /bil/proj/rf1hillman/HOLiS_NPBB328_Cortex/Slab6/2025_08_22_HOLiS_NPBB328_Cortex_Slab06/NPBB328-Cortex-Slab06-run388-z05-y000-darkFrames_HiCAM FLUO_1875-ST-272.fli.zst
Input file: /bil/proj/rf1hillman/HOLiS_NPBB328_Cortex/Slab6/2025_08_22_HOLiS_NPBB328_Cortex_Slab06/NPBB328-Cortex-Slab06-run239-z04-y000-darkFrames_HiCAM FLUO_1875-ST-272.fli.zst
job_script_path: /bil/proj/rf1hillman/results/NPBB328_Cortex/Slab6/correction_omehans/job_NPBB328-Cortex-Slab06-run239-z04-y000-darkFrames_HiCAM FLUO_1875-ST-272.sh
Submitted batch job 976076
Submitted job for /bil/proj/rf1hillman/HOLiS_NPBB32

In [15]:
# We need to load a bg for every z
for z in range(1,z_slices+1):
    bg_file_name = f'NPBB328-Cortex-Slab06-run*-z{z:02d}-y000-darkFrames_HiCAM FLUO_1875-ST-272.fli.zst'
    bg_path_list = glob(os.path.join(output_dir_omehans, bg_file_name))
    out_path = output_dir + f'bg_mask_slab{slab}_z{z:02d}.npy'
    if os.path.exists(out_path):
        print(f'Skipping z{z:02}')
        continue
    if bg_path_list==[]:
        print(f"Background file not found for z={z}: {sample_dir}")
        continue  
    elif len(bg_path_list) > 1:
        print('More than 1 file matching desired file name pattern.')
    else:
        bg_path = bg_path_list[0]
    
    bg = read_omehans(bg_path)
    print(f'Bg shape for z={z} is {bg.shape}')
    bg = bg.compute()
    bg = bg.astype(np.float32)
    bg_mask = np.median(bg, axis=0)  # shape: (Z, Y)
    print(f'Bg mask shape for z={z} is {bg_mask.shape}')
    out_path = output_dir + f'bg_mask_slab{slab}_z{z}.npy'
    np.save(out_path, bg_mask)
    print(f'Saved bg mask for z= {z} to {out_path}')

Background file not found for z=1: /bil/proj/rf1hillman/HOLiS_NPBB328_Cortex/Slab6/2025_08_22_HOLiS_NPBB328_Cortex_Slab06/
Bg shape for z=2 is (25590, 1024, 1280)


KeyboardInterrupt: 

### Flat field correction load

In [14]:
# ## ------- Flat field correction ------- ##

ffNuclei_path = correction_path + 'externalEpoxy-FF-run001-LEDblue_HiCAM FLUO_1875-ST-272.fli'
ffNuclei_path = os.path.join(os.path.dirname(correction_path), os.path.basename(ffNuclei_path).replace('.fli', '.fli_ZARR_OUT'))
ffNuclei = read_omehans(ffNuclei_path)
print("ffNuclei shape", ffNuclei.shape)
ffNuclei = ffNuclei.compute()
ffNuclei = ffNuclei.astype(np.float32)
FFb_nuclei = np.median(ffNuclei, axis=0)  # shape: (Y, Z)
print("FFb_nuclei shape", FFb_nuclei.shape)
np.save(correction_path + "FFb_nuclei1.npy", FFb_nuclei)

ffNuclei shape (1000, 1024, 1280)
FFb_nuclei shape (1024, 1280)


In [15]:
# --- Load background flat field volume ---
ffbgNuclei_path = correction_path + "externalEpoxy-Laser-run001-darkFrames_HiCAM FLUO_1875-ST-272.fli"
ffbgNuclei_path = os.path.join(os.path.dirname(correction_path), os.path.basename(ffbgNuclei_path).replace('.fli', '.fli_ZARR_OUT'))
ffbgNuclei = read_omehans(ffbgNuclei_path)
print("ffbgNuclei shape", ffbgNuclei.shape)
ffbgNuclei = ffbgNuclei.compute()
ffbgNuclei = ffbgNuclei.astype(np.float32)

# --- Compute flat field background mask ---
FF_BG_nuclei = np.median(ffbgNuclei, axis=0)  # shape: (Y, X)
print("FF_BG_nuclei shape", FF_BG_nuclei.shape)
np.save(correction_path + "FF_BG_nuclei.npy", FF_BG_nuclei)

FF_BG_nuclei_mask = FF_BG_nuclei - 1024.0  # subtract 2^10
FFb_nuclei = FFb_nuclei - FF_BG_nuclei_mask  # shape: (Y, X)
print("FFb_nuclei.shape", FFb_nuclei.shape)
np.save(correction_path + "FFb_nuclei2.npy", FFb_nuclei)

ffbgNuclei shape (1000, 1024, 1280)
FF_BG_nuclei shape (1024, 1280)
FFb_nuclei.shape (1024, 1280)


In [16]:
# --- Normalize flat field ---
FF_nuc_norm = FFb_nuclei / np.median(FFb_nuclei)
print("FF_nuc_norm.shape", FF_nuc_norm.shape)
np.save(correction_path + "FF_nuc_norm.npy", FF_nuc_norm)

FF_nuc_norm.shape (1024, 1280)


### Background correction load

In [5]:
corrbgNuclei_path = correction_path + "NPBB328-corrections-run001-darkFrames_HiCAM FLUO_1875-ST-272.fli"
corrbgNuclei_path = os.path.join(os.path.dirname(correction_path), os.path.basename(corrbgNuclei_path).replace('.fli', '.fli_ZARR_OUT'))
print("reading corrbgNuclei")
corrbgNuclei = read_omehans(corrbgNuclei_path)
print("corrbgNuclei shape", corrbgNuclei.shape)
Corr_BG_nuclei = np.median(corrbgNuclei.astype(np.float32), axis=0)
print("Corr_BG_nuclei shape", Corr_BG_nuclei.shape)
np.save(correction_path + "Corr_BG_nuclei", Corr_BG_nuclei)

reading corrbgNuclei
corrbgNuclei shape (1000, 1024, 1280)
Corr_BG_nuclei shape (1024, 1280)


/bil/users/psimko/.conda/envs/stack_to_multiscale_ngff/lib/python3.8/site-packages/dask/array/core.py:1713: FutureWarning: The `numpy.save` function is not implemented by Dask array. You may want to use the da.map_blocks function or something similar to silence this warning. Your code may stop working in a future release.
  warnings.warn(


In [6]:
Corr_BG_nuclei_mask = Corr_BG_nuclei - 1024.0

### Laser correction load

In [7]:
# --- Prepare arrays to store masks ---
lasers = [488, 561, 594, 660]
n_lasers = len(lasers)
Y, X = Corr_BG_nuclei_mask.shape
POWELL_NUC_MASK = np.zeros((n_lasers, Y, X), dtype=np.float32)
print("POWELL_NUC_MASK shape", POWELL_NUC_MASK.shape)

NameError: name 'Corr_BG_nuclei_mask' is not defined

In [27]:
# --- Loop over lasers ---
for i, las in enumerate(lasers):
    print("laser", las)
    # Locate file
    Powell_Nuclei_path = glob(correction_path + f'NPBB328-corrections-epoxy-run*{las}nm_HiCAM FLUO_1875-ST-272.fli_ZARR_OUT')[0]
    Powell_Nuclei_path = os.path.join(os.path.dirname(correction_path), os.path.basename(Powell_Nuclei_path))

    # Read HiCAM stacks
    print("reading Powell_Nuclei")
    Powell_Nuclei = read_omehans(Powell_Nuclei_path)

    Powell_Nuclei = Powell_Nuclei.astype(np.float32)

    # Median projection and background subtraction
    Powell_Nuclei_mask = np.median(Powell_Nuclei, axis=0) - Corr_BG_nuclei_mask
    print("Powell_Nuclei_mask shape", Powell_Nuclei_mask.shape)

    # Save masks
    POWELL_NUC_MASK[i, :, :] = Powell_Nuclei_mask

np.save(correction_path + "POWELL_NUC_MASK.npy", POWELL_NUC_MASK)

laser 488
reading Powell_Nuclei
Powell_Nuclei_mask shape (1024, 1280)
laser 561
reading Powell_Nuclei
Powell_Nuclei_mask shape (1024, 1280)
laser 594
reading Powell_Nuclei
Powell_Nuclei_mask shape (1024, 1280)
laser 660
reading Powell_Nuclei
Powell_Nuclei_mask shape (1024, 1280)


### Divide laser correction by flat field correction

In [29]:
POWELL_NUC_MASK = np.load(correction_path + "POWELL_NUC_MASK.npy")
FF_nuc_norm = np.load(correction_path + "FF_nuc_norm.npy")

Y, X = FF_nuc_norm.shape
print("Y, X", Y, X)

# Output array
POWELL_NUC_MASK_FF_norm = np.zeros((n_lasers, Y, X), dtype=np.float32)
print("POWELL_NUC_MASK_FF_norm shape", POWELL_NUC_MASK_FF_norm.shape)

for i, las in enumerate(lasers):
    nuc_temp = POWELL_NUC_MASK[i, :, :] / FF_nuc_norm
    print("nuc_temp shape", nuc_temp.shape)

    cropped_nuc = nuc_temp[250:-250, 250:-250]
    nuc_temp -= np.min(cropped_nuc)
    
    max_val = np.max(cropped_nuc)
    if max_val > 0:
        nuc_temp /= max_val
    else:
        print(f"Warning: max(cropped_nuc) = {max_val} at laser index {i}, skipping normalization")
        nuc_temp[:] = 0.0

    POWELL_NUC_MASK_FF_norm[i, :, :] = nuc_temp

np.save(correction_path + "POWELL_NUC_MASK_FF_norm.npy", POWELL_NUC_MASK_FF_norm)

Y, X 1024 1280
POWELL_NUC_MASK_FF_norm shape (4, 1024, 1280)
7810.0884
nuc_temp shape (1024, 1280)
7975.948
12838.279
nuc_temp shape (1024, 1280)
13196.143
3753.2039
nuc_temp shape (1024, 1280)
3733.8958
2723.9146
nuc_temp shape (1024, 1280)
2779.994


In [30]:
# --- Nuclei correction from first laser only ---
Laser_correction_Nuclei = POWELL_NUC_MASK_FF_norm[0, :, :]  # shape: (Y, X)


# Compute smooth mask (column stripes) from nuclei pattern
median_profile = np.median(Laser_correction_Nuclei, axis=0)  # shape: (X,)

print("median_profile", median_profile.shape)

smoothed_profile = gaussian_filter1d(median_profile, sigma=Laser_correction_Nuclei.shape[1] / 10)
Laser_correction_Nuclei_pattern = np.outer(np.ones(Laser_correction_Nuclei.shape[0]), median_profile)
Laser_correction_Nuclei_pattern /= smoothed_profile  # broadcasting column-wise

Laser_correction_Nuclei_pattern = ((Laser_correction_Nuclei_pattern - 1) / 3) + 1

print("Laser_correction_Nuclei_pattern.shape", Laser_correction_Nuclei_pattern.shape)

np.save(correction_path + "Laser_correction_Nuclei_pattern.npy", Laser_correction_Nuclei_pattern)

median_profile (1280,)
Laser_correction_Nuclei_pattern.shape (1024, 1280)


### Laser correction load 2 - october 2025

In [10]:
slab = 'Slab6'
z = '01'
#corr_path = f'/bil/proj/rf1hillman/HOLiS_NPBB328_Cortex/{slab}/{slab}/2025_08_22_HOLiS_NPBB328_Cortex_{slab}_corrections{z}/'
#mat_path = os.path.join(corr_path, f"2025_08_22_HOLiS_NPBB328_Cortex_{slab}_corrections{z}.mat")
corr_path = '/bil/proj/rf1hillman/HOLiS_NPBB328_Cortex/Slab6/2025_08_22_HOLiS_NPBB328_Cortex_Slab06/'
#mat_path = os.path.join(corr_path, f"2025_08_22_HOLiS_NPBB328_Cortex_{slab}_corrections{z}.mat")
mat_path = os.path.join(corr_path, f"NPBB328_Cortex_Slab06_run1031_z13_y076_Exc_488nm_561nm_594nm_660nm_info.mat")
data = loadmat(mat_path, squeeze_me=True, struct_as_record=False)

In [11]:
print(data)

{'__header__': b'MATLAB 5.0 MAT-file, Platform: PCWIN64, Created on: Sun Aug 24 04:42:06 2025', '__version__': '1.0', '__globals__': [], 'info': <scipy.io.matlab._mio5_params.mat_struct object at 0x7fb6de1a51c0>, '__function_workspace__': array([[ 0,  1, 73, ...,  0,  0,  0]], dtype=uint8)}


In [12]:
path='/bil/proj/rf1hillman/2024_07_29_AI7_EH5k_human_finalMarkerCombination_100mm/code_Matlab/SimulationMatrices_equalPower_firstHemibrain.mat'
data = loadmat(path, squeeze_me=True, struct_as_record=False)

In [16]:
print(data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'Flch', 'excitation_efficiency'])


## Colors

### Background load

In [ ]:
# We need to load a bg for every z
for z in range(1,z_slices+1):
    bg_file_name = f'NPBB328-Cortex-Slab06-run*-z{z:02d}-y000-darkFrames_HiCAM FLUO_1875-ST-272.fli.zst'
    bg_path_list = glob(os.path.join(output_dir_omehans, bg_file_name))
    if bg_path_list==[]:
        print(f"Background file not found for z={z}: {sample_dir}")
        continue  
    elif len(bg_path_list) > 1:
        print('More than 1 file matching desired file name pattern.')
    else:
        bg_path = bg_path_list[0]
    
    bg = read_omehans(bg_path)
    print(f'Bg shape for z={z} is {bg.shape}')
    bg = bg.compute()
    bg = bg.astype(np.float32)
    bg_mask = np.median(bg, axis=0)  # shape: (Z, Y)
    print(f'Bg mask shape for z={z} is {bg_mask.shape}')
    out_path = output_dir + f'bg_mask_slab{slab}_z{z}.npy'
    np.save(out_path, bg_mask)
    print(f'Saved bg mask for z= {z} to {out_path}')

### Flat field correction load

In [32]:
# ## ------- Flat field correction ------- ##

ffColors_path = correction_path + 'externalEpoxy-FF-run001-LEDblue_HiCAM FLUO_1875-ST-088.fli'
ffColors_path = os.path.join(os.path.dirname(correction_path), os.path.basename(ffColors_path).replace('.fli', '.fli_ZARR_OUT'))
ffColors = read_omehans(ffColors_path)
print("ffColors shape", ffColors.shape)
ffColors = ffColors.compute()
ffColors = ffColors.astype(np.float32)
FFb_colors = np.median(ffColors, axis=0)  # shape: (Y, Z)
print("FFb_colors shape", FFb_colors.shape)
np.save(correction_path + "FFb_colors1.npy", FFb_colors)

ffColors shape (1000, 1024, 1280)
FFb_colors shape (1024, 1280)


In [33]:
# --- Load background flat field volume ---
ffbgColors_path = correction_path + "externalEpoxy-Laser-run001-darkFrames_HiCAM FLUO_1875-ST-088.fli"
ffbgColors_path = os.path.join(os.path.dirname(correction_path), os.path.basename(ffbgColors_path).replace('.fli', '.fli_ZARR_OUT'))
ffbgColors = read_omehans(ffbgColors_path)
print("ffbgColors shape", ffbgColors.shape)
ffbgColors = ffbgColors.compute()
ffbgColors = ffbgColors.astype(np.float32)

# --- Compute flat field background mask ---
FF_BG_colors = np.median(ffbgColors, axis=0)  # shape: (Y, X)
print("FF_BG_colors shape", FF_BG_colors.shape)
np.save(correction_path + "FF_BG_colors.npy", FF_BG_colors)

FF_BG_colors_mask = FF_BG_colors - 1024.0  # subtract 2^10
FFb_colors = FFb_colors - FF_BG_colors_mask  # shape: (Y, X)
print("FFb_colors.shape", FFb_colors.shape)
np.save(correction_path + "FFb_colors2.npy", FFb_colors)

ffbgColors shape (1000, 1024, 1280)
FF_BG_colors shape (1024, 1280)
FFb_colors.shape (1024, 1280)


In [34]:
# --- Normalize flat field ---
FF_colors_norm = FFb_colors / np.median(FFb_colors)
print("FF_colors_norm.shape", FF_colors_norm.shape)
np.save(correction_path + "FF_colors_norm.npy", FF_colors_norm)

FF_colors_norm.shape (1024, 1280)


### Background correction load

In [36]:
corrbgColors_path = correction_path + "NPBB328-corrections-run001-darkFrames_HiCAM FLUO_1875-ST-088.fli"
corrbgColors_path = os.path.join(os.path.dirname(correction_path), os.path.basename(corrbgColors_path).replace('.fli', '.fli_ZARR_OUT'))
print("reading corrbgColors")
corrbgColors = read_omehans(corrbgColors_path)
print("corrbgColors shape", corrbgColors.shape)
Corr_BG_colors = np.median(corrbgColors.astype(np.float32), axis=0)
print("Corr_BG_colors shape", Corr_BG_colors.shape)
np.save(correction_path + "Corr_BG_colors", Corr_BG_colors)

reading corrbgColors
corrbgColors shape (1000, 1024, 1280)
Corr_BG_colors shape (1024, 1280)


/bil/users/psimko/.conda/envs/stack_to_multiscale_ngff/lib/python3.8/site-packages/dask/array/core.py:1713: FutureWarning: The `numpy.save` function is not implemented by Dask array. You may want to use the da.map_blocks function or something similar to silence this warning. Your code may stop working in a future release.
  warnings.warn(


In [37]:
Corr_BG_colors_mask = Corr_BG_colors - 1024.0

### Laser correction load

In [40]:
# --- Prepare arrays to store masks ---
lasers = [488, 561, 594, 660]
n_lasers = len(lasers)
Y, X = Corr_BG_colors_mask.shape
POWELL_COLORS_MASK = np.zeros((n_lasers, Y, X), dtype=np.float32)
print("POWELL_COLORS_MASK shape", POWELL_COLORS_MASK.shape)

POWELL_COLORS_MASK shape (4, 1024, 1280)


In [41]:
# --- Loop over lasers ---
for i, las in enumerate(lasers):
    print("laser", las)
    # Locate file
    Powell_Colors_path = glob(correction_path + f'NPBB328-corrections-epoxy-run*{las}nm_HiCAM FLUO_1875-ST-088.fli_ZARR_OUT')[0]
    Powell_Colors_path = os.path.join(os.path.dirname(correction_path), os.path.basename(Powell_Colors_path))

    # Read HiCAM stacks
    print("reading Powell_Colors")
    Powell_Colors = read_omehans(Powell_Colors_path)

    Powell_Colors = Powell_Colors.astype(np.float32)

    # Median projection and background subtraction
    Powell_Colors_mask = np.median(Powell_Colors, axis=0) - Corr_BG_colors_mask
    print("Powell_Colors_mask shape", Powell_Colors_mask.shape)

    # Save masks
    POWELL_COLORS_MASK[i, :, :] = Powell_Colors_mask

np.save(correction_path + "POWELL_COLORS_MASK.npy", POWELL_COLORS_MASK)

laser 488
reading Powell_Colors
Powell_Colors_mask shape (1024, 1280)
laser 561
reading Powell_Colors
Powell_Colors_mask shape (1024, 1280)
laser 594
reading Powell_Colors
Powell_Colors_mask shape (1024, 1280)
laser 660
reading Powell_Colors
Powell_Colors_mask shape (1024, 1280)


### Divide laser correction by flat field correction

In [42]:
POWELL_COLORS_MASK = np.load(correction_path + "POWELL_COLORS_MASK.npy")
FF_nuc_norm = np.load(correction_path + "FF_colors_norm.npy")

Y, X = FF_colors_norm.shape
print("Y, X", Y, X)

# Output array
POWELL_COLORS_MASK_FF_norm = np.zeros((n_lasers, Y, X), dtype=np.float32)
print("POWELL_COLORS_MASK_FF_norm shape", POWELL_COLORS_MASK_FF_norm.shape)

for i, las in enumerate(lasers):
    colors_temp = POWELL_COLORS_MASK[i, :, :] / FF_colors_norm
    print("colors_temp shape", colors_temp.shape)

    cropped_colors = colors_temp[250:-250, 250:-250]
    colors_temp -= np.min(cropped_colors)
    
    max_val = np.max(cropped_colors)
    if max_val > 0:
        colors_temp /= max_val
    else:
        print(f"Warning: max(cropped_colors) = {max_val} at laser index {i}, skipping normalization")
        colors_temp[:] = 0.0

    POWELL_COLORS_MASK_FF_norm[i, :, :] = colors_temp

np.save(correction_path + "POWELL_COLORS_MASK_FF_norm.npy", POWELL_COLORS_MASK_FF_norm)

Y, X 1024 1280
POWELL_COLORS_MASK_FF_norm shape (4, 1024, 1280)
colors_temp shape (1024, 1280)
colors_temp shape (1024, 1280)
colors_temp shape (1024, 1280)
colors_temp shape (1024, 1280)


In [46]:
# --- Colors correction from respective lasers only ---

# Output array
Laser_correction_Colors_pattern = np.zeros((n_lasers, Y, X), dtype=np.float32)

for i, las in enumerate(lasers):
    Laser_correction_Colors = POWELL_COLORS_MASK_FF_norm[i, :, :]  # shape: (Y, X)

    # Compute smooth mask (column stripes) from nuclei pattern
    median_profile = np.median(Laser_correction_Colors, axis=0)  # shape: (X,)

    print("median_profile", median_profile.shape)

    smoothed_profile = gaussian_filter1d(median_profile, sigma=Laser_correction_Colors.shape[1] / 10)
    Laser_correction_Colors_pattern[i,:,:] = np.outer(np.ones(Laser_correction_Colors.shape[0]), median_profile)
    Laser_correction_Colors_pattern[i,:,:] /= smoothed_profile  # broadcasting column-wise

    Laser_correction_Colors_pattern[i,:,:] = ((Laser_correction_Colors_pattern[i,:,:] - 1) / 3) + 1
    
print("Laser_correction_Colors_pattern has shape", Laser_correction_Colors_pattern.shape)

np.save(correction_path + "Laser_correction_Colors_pattern.npy", Laser_correction_Colors_pattern)

median_profile (1280,)
median_profile (1280,)
median_profile (1280,)
median_profile (1280,)
Laser_correction_Colors_pattern has shape (4, 1024, 1280)
